# Cafeteria wait-time workflow
Working notebook for the FDE assignment. All data is synthetic and reproducible (`python3 generate_data.py`, seed 10252).


## 1. Stakeholders, workflow and KPI
Students, cafeteria manager, kitchen staff and handover staff are affected. The workflow is placed -> accepted -> ready -> handed over. The KPI is percent of completed orders handed over within 12 minutes.


## 2. Retrieval and profiling of the three sources
CSV file export (orders), JSON configuration (shifts and target), and a SQL query on the staff roster database. Raw sources are only read, never changed.


In [1]:
from pathlib import Path
from collections import Counter
import csv, json, sqlite3
ROOT = Path('.')

# Source 1: CSV export
with open(ROOT/'data/raw/orders.csv') as f: orders = list(csv.DictReader(f))
# Source 2: JSON configuration
config = json.loads((ROOT/'data/raw/shift_config.json').read_text())
# Source 3: SQL query (read-only connection), same query the pipeline runs
manifest = json.loads((ROOT/'data/raw/manifest.json').read_text())
conn = sqlite3.connect(f"file:{ROOT/'data/raw/staff_roster.db'}?mode=ro", uri=True)
roster = conn.execute('''SELECT roster_date, shift, staff_scheduled, staff_present
                         FROM staff_roster WHERE roster_date BETWEEN ? AND ? ORDER BY roster_date, shift''',
                      (manifest['staff_roster_db']['period_start'], manifest['staff_roster_db']['period_end'])).fetchall()
conn.close()
print('orders rows:', len(orders), '| manifest says', manifest['orders_csv']['row_count'])
print('config: target', config['service_target_minutes'], 'min; shifts', list(config['shifts']))
print('roster rows:', len(roster), '| manifest says', manifest['staff_roster_db']['row_count'])


orders rows: 2283 | manifest says 2283
config: target 12 min; shifts ['breakfast', 'lunch', 'dinner']
roster rows: 21 | manifest says 21


In [2]:
# Profile: orders
print('orders by shift:', dict(Counter(r['shift'] for r in orders)))
print('missing order_ts:', sum(not r['order_ts'] for r in orders))
print('duplicate order_id rows:', len(orders) - len({r['order_id'] for r in orders}))
print('cancelled values:', dict(Counter(r['cancelled'] for r in orders)))
print('issue codes:', dict(Counter(r['issue_code'] or '(none)' for r in orders)))
# Profile: roster
short = [r for r in roster if r[3] < r[2]]
print('roster days:', len({r[0] for r in roster}), '| short-staffed shift-days:', len(short))
for r in short: print('  ', r)


orders by shift: {'breakfast': 601, 'lunch': 875, 'dinner': 806, 'brunch': 1}
missing order_ts: 1
duplicate order_id rows: 1
cancelled values: {'false': 2212, 'true': 70, 'yes': 1}
issue codes: {'(none)': 2086, 'kitchen_rework': 73, 'stockout': 65, 'payment': 59}
roster days: 7 | short-staffed shift-days: 5
   ('2026-09-07', 'breakfast', 4, 3)
   ('2026-09-08', 'lunch', 5, 4)
   ('2026-09-09', 'breakfast', 4, 3)
   ('2026-09-09', 'dinner', 4, 3)
   ('2026-09-13', 'dinner', 4, 3)


## 3. Repeatable validation, transformation and model
Run the pipeline to validate, quarantine unsafe rows, derive wait and compliance, and load SQLite.


In [3]:
%run pipeline.py


2026-09-25 01:26:32,434 INFO run 86fc4a92 started


2026-09-25 01:26:32,462 INFO Retrieved 2283 CSV rows (sha256 3b787ac0cd47..., matches manifest), JSON config with 3 shifts, and 21 roster rows via SQL


2026-09-25 01:26:32,482 INFO Accepted 2275 rows; rejected 8; reasons {'exact_duplicate_dropped': 1, 'missing_core_timestamp': 1, 'completed_without_handover': 1, 'timestamp_sequence': 1, 'invalid_shift': 1, 'shift_window_mismatch': 1, 'invalid_cancelled': 1, 'invalid_timestamp': 1}


2026-09-25 01:26:32,527 INFO run 86fc4a92 finished; KPI 57.55%


{
  "project_kpi": "percent of completed orders handed over within 12 minutes",
  "project_kpi_value_pct": 57.55,
  "completed_orders": 2205,
  "average_wait_minutes": 11.44,
  "average_stage_minutes": {
    "accept": 1.03,
    "prepare": 8.74,
    "handover": 1.66
  },
  "cancellation_rate_pct": 3.08,
  "issue_rate_pct": 8.66,
  "within_target_pct_by_issue": {
    "no_issue": 58.78,
    "with_issue": 44.44
  },
  "shift_metrics": [
    {
      "shift": "breakfast",
      "completed_orders": 580,
      "within_target_pct": 52.24,
      "avg_wait_minutes": 11.92,
      "avg_accept_min": 1.08,
      "avg_prep_min": 9.18,
      "avg_handover_min": 1.72,
      "avg_staff_present": 3.71,
      "orders_per_staff_hour": 11.44
    },
    {
      "shift": "dinner",
      "completed_orders": 777,
      "within_target_pct": 56.63,
      "avg_wait_minutes": 11.43,
      "avg_accept_min": 1.03,
      "avg_prep_min": 8.78,
      "avg_handover_min": 1.65,
      "avg_staff_present": 3.71,
      "order

## 4. Metric output


In [4]:
summary = json.loads((ROOT/'data/processed/summary.json').read_text())
quality = json.loads((ROOT/'data/processed/quality_report.json').read_text())
print('KPI:', summary['project_kpi_value_pct'], '%')
print('stage minutes:', summary['average_stage_minutes'])
print('by staffing:', summary['within_target_by_staffing'])
print('quarantined:', quality['rejected_rows'], quality['rejections_by_reason'])


KPI: 57.55 %
stage minutes: {'accept': 1.03, 'prepare': 8.74, 'handover': 1.66}
by staffing: {'fully_staffed': {'shift_days': 16, 'completed_orders': 1698, 'within_target_pct': 59.54}, 'short_staffed': {'shift_days': 5, 'completed_orders': 507, 'within_target_pct': 50.89}}
quarantined: 8 {'exact_duplicate_dropped': 1, 'missing_core_timestamp': 1, 'completed_without_handover': 1, 'timestamp_sequence': 1, 'invalid_shift': 1, 'shift_window_mismatch': 1, 'invalid_cancelled': 1, 'invalid_timestamp': 1}


## 5. Assumptions and limitations
The 12-minute target and synthetic distributions require stakeholder validation. The ~15% slower preparation on short-staffed shifts is a generator assumption. Menu complexity, queue length and mid-shift staffing changes are omitted. No unsafe timestamps are imputed. The next step is replacing synthetic inputs with real exports and the real roster database, and adding data-freshness alerts.
